# CMPE492 — Wav2Lip External Baseline (Stage 4)

**Purpose:** Add a strong external SOTA baseline. Wav2Lip (ACM MM 2020) is the standard high-LSE-C lip-sync reference. This directly addresses the biggest gap: comparing our hybrid against an established end-to-end method, not just its own ablated parts.

**Fair-comparison protocol (identical cross-audio mapping as Stage 2/3):**
- For each identity: `face = raw[idN]` video, `audio = id(N+1)` cross audio
- e.g. id01 face + id02 audio → wav2lip/id01.mp4
- This matches exactly what Hybrid-LP and LP-only received.

**Input:** `hdtf_prepared.zip` (Stage 1: raw/ videos + audio/)
**Output:** `wav2lip_videos.zip` — one Wav2Lip video per identity, ready to add to Batch Evaluation

---
**Run order:** 1 → 2 → 3 → 4 → 5

## Cell 1 — Clone Wav2Lip + Install + Patch

In [ ]:
import subprocess, sys, os

# Clone canonical Rudrabha/Wav2Lip (the reference implementation)
if not os.path.exists('/content/Wav2Lip'):
    subprocess.run('git clone https://github.com/Rudrabha/Wav2Lip.git /content/Wav2Lip',
                   shell=True, check=True)
    print('✅ Cloned Wav2Lip')
else:
    print('✅ Wav2Lip already present')

os.chdir('/content/Wav2Lip')

# Install deps (keep modern librosa, patch the API call instead of downgrading)
for pkg in ['librosa', 'numba', 'opencv-python', 'opencv-contrib-python',
            'batch-face', 'gdown']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], capture_output=True)
print('✅ Dependencies installed')

# ── PATCH 1: librosa.filters.mel() positional → keyword args (librosa 0.10+ fix) ──
audio_py = '/content/Wav2Lip/audio.py'
with open(audio_py) as f: src = f.read()
import re
# original: librosa.filters.mel(hp.sample_rate, hp.n_fft, n_mels=..., fmin=..., fmax=...)
src = re.sub(
    r'librosa\.filters\.mel\(hp\.sample_rate,\s*hp\.n_fft,',
    'librosa.filters.mel(sr=hp.sample_rate, n_fft=hp.n_fft,',
    src)
# also handle librosa.load(path, sr) and any stft positional issues are fine
with open(audio_py, 'w') as f: f.write(src)
print('✅ Patched audio.py (librosa mel keyword args)')

# ── PATCH 2: inference.py — guard against librosa.load positional sr if present ──
# (modern librosa wants sr= keyword; Wav2Lip already uses keyword, so usually fine)

import librosa
print(f'librosa: {librosa.__version__}')
import torch
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
print('\n✅ Cell 1 complete')

## Cell 2 — Download Checkpoints (reliable GitHub mirrors)

In [ ]:
import os, subprocess
os.chdir('/content/Wav2Lip')
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('face_detection/detection/sfd', exist_ok=True)

# justinjohn0306 GitHub release mirrors — stable, no Google Drive rate-limits
downloads = [
    ('checkpoints/wav2lip_gan.pth',
     'https://github.com/justinjohn0306/Wav2Lip/releases/download/models/wav2lip_gan.pth'),
    ('checkpoints/wav2lip.pth',
     'https://github.com/justinjohn0306/Wav2Lip/releases/download/models/wav2lip.pth'),
    ('face_detection/detection/sfd/s3fd.pth',
     'https://github.com/justinjohn0306/Wav2Lip/releases/download/models/s3fd.pth'),
]
for dst, url in downloads:
    if os.path.exists(dst) and os.path.getsize(dst) > 1000:
        print(f'✅ {os.path.basename(dst)} already present')
        continue
    print(f'Downloading {os.path.basename(dst)}...')
    r = subprocess.run(f'wget -q "{url}" -O {dst}', shell=True, capture_output=True, text=True)
    if os.path.exists(dst) and os.path.getsize(dst) > 1000:
        print(f'  ✅ {os.path.basename(dst)} ({os.path.getsize(dst)//(1024*1024)} MB)')
    else:
        print(f'  ⚠ Failed: {url}')
        print(f'    Try manual download or alternate mirror.')

print('\n✅ Checkpoints ready')

## Cell 3 — Upload HDTF Data
> Upload `hdtf_prepared.zip` (from Stage 1)

In [ ]:
import os, zipfile, glob
from google.colab import files

HDTF = '/content/hdtf_prepared'
os.makedirs(HDTF, exist_ok=True)

print('Upload hdtf_prepared.zip')
uploaded = files.upload()
for fname, data in uploaded.items():
    if fname.endswith('.zip'):
        with open(f'/content/{fname}', 'wb') as f: f.write(data)
        with zipfile.ZipFile(f'/content/{fname}') as z: z.extractall(HDTF)
        print(f'✅ Extracted {fname}')

raw_videos = {}
for v in sorted(glob.glob(f'{HDTF}/**/raw/*.mp4', recursive=True)):
    stem = os.path.splitext(os.path.basename(v))[0]
    raw_videos[stem] = v
audios = {}
for a in sorted(glob.glob(f'{HDTF}/**/audio/*.wav', recursive=True)):
    stem = os.path.splitext(os.path.basename(a))[0]
    audios[stem] = a

print(f'\nRaw videos: {len(raw_videos)} → {sorted(raw_videos.keys())}')
print(f'Audios: {len(audios)} → {sorted(audios.keys())}')

## Cell 4 — Batch Wav2Lip (cross-audio, with resume)

In [ ]:
import os, subprocess, shlex, glob, json, time, shutil
os.chdir('/content/Wav2Lip')

OUT = '/content/wav2lip_videos'
os.makedirs(OUT, exist_ok=True)
PROGRESS = '/content/wav2lip_progress.json'

# SAME cross-audio mapping as Stage 2/3 (cyclic)
def cross_audio_id(stem):
    n = int(stem.replace('id',''))
    return f'id{(n % 10) + 1:02d}'

if os.path.exists(PROGRESS):
    with open(PROGRESS) as f: completed = json.load(f)
else:
    completed = {}

stems = sorted(raw_videos.keys())
print('Cross-audio mapping:')
for s in stems:
    print(f'  {s} face + {cross_audio_id(s)} audio')
print()

for stem in stems:
    ca_id = cross_audio_id(stem)
    final_out = f'{OUT}/{stem}.mp4'
    if stem in completed and os.path.exists(final_out):
        print(f'✓ {stem} (audio {ca_id}) already done')
        continue
    if ca_id not in audios:
        print(f'⚠ {stem}: cross audio {ca_id} missing, skipping')
        continue

    face = raw_videos[stem]
    audio = audios[ca_id]
    print(f'\n{"="*55}\n  {stem} ← {ca_id} audio\n{"="*55}')
    t0 = time.time()

    r = subprocess.run(
        f'python inference.py '
        f'--checkpoint_path checkpoints/wav2lip_gan.pth '
        f'--face {shlex.quote(face)} '
        f'--audio {shlex.quote(audio)} '
        f'--outfile {shlex.quote(final_out)} '
        f'--pads 0 15 0 0 '
        f'--resize_factor 1',
        shell=True, capture_output=True, text=True)
    print(f'  ⏱ {time.time()-t0:.1f}s')

    if os.path.exists(final_out) and os.path.getsize(final_out) > 1000:
        completed[stem] = {'face': face, 'cross_audio': ca_id}
        with open(PROGRESS, 'w') as f: json.dump(completed, f, indent=2)
        print(f'  ✅ {stem}.mp4 (face={stem}, audio={ca_id})')
    else:
        print(f'  ⚠ failed: {r.stderr[-400:]}')

print(f'\n{"="*55}')
print(f'  ✅ Wav2Lip baseline: {len(completed)}/{len(stems)} identities')
print(f'{"="*55}')

## Cell 5 — Package

In [ ]:
import shutil, os, glob
from google.colab import files

OUT = '/content/wav2lip_videos'
vids = sorted(glob.glob(f'{OUT}/*.mp4'))
print(f'Wav2Lip videos: {len(vids)}')
for v in vids:
    print(f'  {os.path.basename(v)} ({os.path.getsize(v)//1024} KB)')

shutil.make_archive('/content/wav2lip_videos', 'zip', OUT)
sz = os.path.getsize('/content/wav2lip_videos.zip')//(1024*1024)
print(f'\n✅ Packaged: wav2lip_videos.zip ({sz} MB)')
files.download('/content/wav2lip_videos.zip')

print('''
─────────────────────────────────────────────────────────
NEXT: Add Wav2Lip to your Batch Evaluation
─────────────────────────────────────────────────────────
1. In the Batch Evaluation notebook, extract wav2lip_videos.zip
   into batch_eval/wav2lip/  (alongside hybrid_lp/, lp_only/)
2. In Cell 2, change the CONFIGS dict to:

   CONFIGS = {
       'hybrid_lp': list_videos('hybrid_lp'),
       'lp_only':   list_videos('lp_only'),
       'wav2lip':   list_videos('wav2lip'),
   }

3. Re-run the evaluation → you get Wav2Lip's LSE-C/CSIM alongside yours.
''')